# PROJECT: BERT for Text Classification

Reach for this when you need: 
- Complete pure PyTorch boilerplate for fine-tuning Transformers.
- Reference for using HuggingFace `datasets` with custom training loops.
- Implementing Mixed Precision training for BERT.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Data Pipeline (HuggingFace Datasets)

| Step | Action | Logic |
| :--- | :--- | :--- |
| `load_dataset` | Fetch IMDb or SST-2 | Industry standard for NLP benchmarks |
| `tokenize` | map function | efficient batch-wise tokenization |
| `set_format` | Convert to PyTorch | Ensuring compatibility with DataLoader |

In [ ]:
dataset = load_dataset("sst2")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_fn(examples):
    return tokenizer(examples['sentence'], padding='max_length', truncation=True)

tokenized_ds = dataset.map(tokenize_fn, batched=True)
tokenized_ds.set_format(type='torch', columns=['input_ids', 'token_type_ids', 'attention_mask', 'label'])

train_loader = DataLoader(tokenized_ds['train'], batch_size=16, shuffle=True)
test_loader = DataLoader(tokenized_ds['validation'], batch_size=16)

## 2. Model & Optimization

Using `AutoModelForSequenceClassification` which automatically adds a classification head on top of the BERT encoder.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(device)

optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler() # Handling mixed precision
criterion = nn.CrossEntropyLoss()

## 3. Custom Training Loop (Pure PyTorch)

Manual loop for maximum visibility into gradient flows and AMP handling.

In [ ]:
model.train()
for batch in tqdm(train_loader):
    inputs = {k: v.to(device) for k, v in batch.items() if k != 'label'}
    labels = batch['label'].to(device)
    
    optimizer.zero_grad()
    
    with torch.cuda.amp.autocast(): # Mixed precision
        outputs = model(**inputs)
        loss = criterion(outputs.logits, labels)
        
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

### Common Pitfalls
- **Learning Rate**: For Transformers, use VERY small learning rates (e.g. 5e-5, 2e-5). Larger values will cause the model to forget pretrained features immediately.
- **Padding**: Padding to `max_length=512` is memory-heavy. Use dynamic padding (padding to max length in a batch) for faster loops.
- **Loss Function**: `AutoModel` can return loss directly if labels are passed (`model(**inputs, labels=labels)`). This ensures the correct loss for the architecture.

### Key Takeaways
- `Mixed Precision` (AMP) is essential for training Transformers on consumer GPUs efficiently.
- HuggingFace `datasets` handles disk-caching automatically, preventing RAM bottlenecks.
- Fine-tuning BERT is the industry baseline for any text classification task.